In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI
import os
llm = ChatGoogleGenerativeAI(
    model = "gemini-2.5-flash",
    temperature = 0,
    api_key = os.getenv("GOOGLE_API_KEY")
)

In [2]:
#checking if llm response is working
#basic we use chatprompt template
query = "WHO IS THE CEO OF OPENAI AND META"
result = llm.invoke(query)
print(result)

content='*   The CEO of **OpenAI** is **Sam Altman**.\n*   The CEO of **Meta** (formerly Facebook) is **Mark Zuckerberg**.' additional_kwargs={} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--019bd716-4083-7bd3-a5da-e59c444ee6d1-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 10, 'output_tokens': 286, 'total_tokens': 296, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 254}}


In [3]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human", "Explain to me about {topic}")
])

#llm is defined above
parser = StrOutputParser()
chain  = prompt | llm |  parser
result = chain.invoke({"topic": "machine learning"})
print(result)


Machine Learning (ML) is a fascinating and rapidly evolving field of Artificial Intelligence (AI) that empowers computers to **learn from data without being explicitly programmed**.

Think of it this way: Instead of a programmer writing specific instructions for every possible scenario, they create an algorithm that allows the computer to analyze data, identify patterns, and then make predictions or decisions based on those patterns.

Here's a breakdown of the core concepts:

### The Core Idea: Learning from Experience

Imagine teaching a child to identify a cat. You don't give them a list of rules like "if it has pointy ears AND whiskers AND meows, it's a cat." Instead, you show them many pictures of cats, point to real cats, and say "that's a cat." You also show them pictures of dogs, birds, etc., and say "that's not a cat." Over time, the child learns to recognize a cat on their own, even if they see a breed they've never encountered before.

Machine learning works similarly. We fee

In [4]:
topic_batch = [
    {"topic" : "RAG"},
    {"topic" : "LCEL"},
    {"topic" : "Langchain agents"}
]

result = chain.batch(topic_batch)
print(result)

['RAG stands for **Retrieval Augmented Generation**.\n\nIt\'s a technique designed to enhance the capabilities of Large Language Models (LLMs) by giving them access to up-to-date, external, and domain-specific information beyond what they were originally trained on.\n\nThink of it like this:\n\n*   **A standard LLM** is like a brilliant student who has read a vast library of books (its training data) but can only answer questions based on what it remembers from those books. If you ask it about something very new or very specific to your company\'s internal documents, it might guess, make things up (hallucinate), or say it doesn\'t know.\n*   **RAG** equips that brilliant student with the ability to quickly look up information in a specific, relevant textbook or database *before* answering your question. This ensures the answer is accurate, current, and grounded in facts.\n\n---\n\n### Why is RAG Important? (The Problem it Solves)\n\nLLMs, despite their impressive abilities, have severa

**DOCUMENTS LOADERS**

✅ Best practice (convert once)
documents = list(pdf_loader.lazy_load())
chunks = splitter.split_documents(documents)


Or simpler (recommended):

documents = pdf_loader.load()
chunks = splitter.split_documents(documents)


In [8]:
from langchain_community.document_loaders import (
    DirectoryLoader,
    PyPDFLoader,
    UnstructuredPDFLoader,
    UnstructuredWordDocumentLoader,
    UnstructuredExcelLoader
)
pdf_loader = DirectoryLoader(
    path="book",
    glob="**/*.pdf",
    loader_cls = PyPDFLoader
)
document = pdf_loader.lazy_load()
#word_loader = DirectoryLoader(
    #path="book",
    #glob = "**/*.docx",
    #loader_cls = UnstructuredWordDocumentLoader
#)
#document = (pdf_loader.load() + word_loader.load())

The Challenge
If you split text randomly:

❌ BAD SPLIT:
Chunk 1: "The transformer architecture revolutionized NLP. It uses self-att"
Chunk 2: "ention mechanisms to process sequences in parallel. This allows..."
The word "attention" is cut in half! 😱

Good splitters respect boundaries (paragraphs, sentences, words):

✅ GOOD SPLIT:
Chunk 1: "The transformer architecture revolutionized NLP. It uses self-attention mechanisms."
Chunk 2: "Self-attention allows the model to process sequences in parallel. This improves speed..."

RecursiveCharacterTextSplitter ⭐
🔰 BEGINNER: The Default Choice
RecursiveCharacterTextSplitter is your go-to splitter for 90% of cases.

How it works:

Tries to split on double newlines (\n\n) → paragraphs
If chunks still too big, splits on single newlines (\n) → lines
If still too big, splits on periods (.) → sentences
If still too big, splits on spaces ( ) → words
Last resort: splits on characters

In [9]:
#CHUNCKING
from langchain_text_splitters import RecursiveCharacterTextSplitter
#we have already store pdf in the foleder documents
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1024,
    chunk_overlap = 120, #10-20 % of chuck size
    length_function = len,
    separators=["\n\n", "\n", " ", ""]
)
chunks = splitter.split_documents(document)
 # Examine first 3 chunks
for i, chunk in enumerate(chunks[:3], 1):
        print(f"{'='*70}")
        print(f"Chunk {i} ({len(chunk.page_content)} chars):")
        print(f"{'='*70}")
        print(chunk.page_content[:300] + "..." if len(chunk.page_content) > 300 else chunk.page_content)
        print()

Chunk 1 (90 chars):
Explorations 
in 
Artificial 
Intelligence 
and 
Machine 
Learning
A 
CRC 
Press 
FreeBook

Chunk 2 (433 chars):
Introduction 
(Prof. 
Roberto 
V. 
Zicari)
1  
-  
Introduction 
to 
Machine 
Learning
2  
-  
The 
Bayesian 
Approach 
to 
Machine 
Learning
3  
-  
A 
Revealing 
Introduction 
to 
Hidden 
Markov 
Models
4  
-  
Introduction 
to 
Reinforcement 
Learning
5  
-  
Deep 
Learning 
for 
Feature 
Represe...

Chunk 3 (240 chars):
READ THE LATEST ON 
ARTIFICIAL INTELLIGENCE AND MACHINE LEARNING
WITH THESE KEY TITLES
VISIT 
WWW.CRCPRE SS.COM/COMPUTE R-SCIE NCE -E NGINE E RING
TO BROWSE OUR FULL RANGE OF TITLES
SAVE  20% AND GET FRE E  SHIPPING WITH DISCOUNT CODE ODB18



7. Chunk Size & Overlap Optimization 📊
🔰 BEGINNER: Rules of Thumb
Recommended Configurations
Content Type	Chunk Size	Overlap	Why
General Text	1000 chars	200 chars	Balanced precision & context
Technical Docs	500-800	100-150	Precision for code/commands
Long Articles	1500-2000	300	More context for narrative
Code	200-400	50-100	Function/class level
FAQs	200-300	30-50	Question-answer pairs
Overlap Guidelines
10-15%: Minimal overlap, saves storage
20%: Sweet spot (recommended)
30%+: Maximum context preservation

In [10]:
!ollama list

NAME                        ID              SIZE      MODIFIED    
gemma3:270m                 e7d36fb2c3b3    291 MB    10 days ago    
mxbai-embed-large:latest    468836162de7    669 MB    10 days ago    
tinyllama:latest            2644915ede35    637 MB    10 days ago    
nomic-embed-text:latest     0a109f422b47    274 MB    10 days ago    
gpt-oss:120b-cloud          569662207105    -         11 days ago    


RETRIVER

UPDATED CODE

In [13]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.chat_models import ChatOllama
from langchain_core.documents import Document

embedding = OllamaEmbeddings(model="mxbai-embed-large:latest")
docs = [
    Document(page_content="LangChain is a framework for LLM applications", metadata={"topic": "langchain"}),
    Document(page_content="RAG combines retrieval with generation", metadata={"topic": "rag"}),
    Document(page_content="Vector databases store embeddings", metadata={"topic": "vectors"}),
    Document(page_content="Transformers use attention mechanisms", metadata={"topic": "transformers"}),
    Document(page_content="FAISS is a similarity search library", metadata={"topic": "vectors"}),
]

faiss_vector = FAISS.from_documents(docs, embedding)
print("✅ Vector store created")


C:\Users\AYUSH SINGH\AppData\Local\Temp\ipykernel_14020\2543719863.py:6: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  embedding = OllamaEmbeddings(model="mxbai-embed-large:latest")


✅ Vector store created


In [ ]:
retriver = faiss_vector.as_retriever(
    search_type = "similarity",
    search_kwargs = {"k" : 3}
)
query = "Explain full form for rag and also tell how does rag works"
results = retriver.invoke(query)
print(f"Query: {query}\n")
print("Results:")
for i, doc in enumerate(results, 1):
    print(f"{i}. {doc.page_content}")
    print(f"   Topic: {doc.metadata['topic']}\n")

Query: Explain full form for rag and also tell how does rag works

Results:
1. RAG combines retrieval with generation
   Topic: rag

2. FAISS is a similarity search library
   Topic: vectors

3. LangChain is a framework for LLM applications
   Topic: langchain



In [ ]:
mmr_retriver = faiss_vector.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 3,           # Final number of results
        "fetch_k": 5,     # Initial pool to select from
        "lambda_mult": 0.5  # 0=diverse, 1=relevant
    }
)
query = "what is vector database"
result = mmr_retriver.invoke(query)
print(result)

TypeError: 'FAISS' object is not callable